In [ ]:
import pandas as pd
import numpy as np
import os
import secrets

## Temporary file cropping for testing
The idea here is for us to be able to quickly iterate through the model using a small subset of the data.  Until the skims are ready, we can just keep using the SANDAG skims, but we need to crop the households, persons, and landuse files to be able to match the SANDAG skims.  Once the skims are ready, the code here will be moot as we can then discard all SANDAG input data.

In [ ]:
# read full input data
input_folder = r".\metro_data"
full_households = pd.read_csv(os.path.join(input_folder, 'households.csv'))
full_persons = pd.read_csv(os.path.join(input_folder, 'persons.csv'))
full_landuse = pd.read_csv(os.path.join(input_folder, 'land_use.csv'))


In [ ]:
# read sandag data so we can crop to the same skims
sandag_landuse = pd.read_csv(os.path.join('sandag_data_cropped', 'land_use.csv'))

In [ ]:
# crop to only the mazs that are in the SANDAG data
cropped_landuse = full_landuse[full_landuse['MAZ'].isin(sandag_landuse['MAZ'])].copy()

# remap maz to taz to match SANDAG so that skims will work
maz_to_taz = sandag_landuse.set_index('MAZ')['TAZ'].to_dict()

cropped_landuse['TAZ'] = cropped_landuse['MAZ'].map(maz_to_taz)

In [ ]:
missed_zones = sandag_landuse[~sandag_landuse['MAZ'].isin(cropped_landuse['MAZ'])]
print(f"Missing {len(missed_zones)} zones in cropped land use data:")
print(missed_zones[['MAZ', 'TAZ']])

In [ ]:
# just grabbing 14 zones from the metro data and renumbering them to match SANDAG
additional_zones = full_landuse[full_landuse['MAZ'].isin(cropped_landuse['MAZ'])].head(missed_zones.shape[0]).copy()
additional_zones[['MAZ', 'TAZ']] = missed_zones[['MAZ', 'TAZ']].values
# append the additional zones to the cropped land use data
cropped_landuse = pd.concat([cropped_landuse, additional_zones], ignore_index=True)


In [ ]:
# obfuscate the employment data
emp_colummns = [col for col in cropped_landuse.columns if col.startswith('EMP')]
for col in emp_colummns:
    cropped_landuse[col] = cropped_landuse[col].apply(lambda x: x * (0.5 + secrets.randbelow(10000) / 10000)).fillna(secrets.randbelow(100)).astype(int)  # scale employment by up to 50% 

In [ ]:
cropped_households = full_households[full_households['MAZ'].isin(cropped_landuse['MAZ'])].copy()
cropped_persons = full_persons[full_persons['household_id'].isin(cropped_households['household_id'])].copy()

# need to map the MAZ and TAZ columns in the households and persons data too
cropped_maz_to_taz = cropped_landuse.set_index('MAZ')[['TAZ']].to_dict()['TAZ']
cropped_households['TAZ'] = cropped_households['MAZ'].map(cropped_maz_to_taz)
print(f"Saving {len(cropped_households)} households and {len(cropped_persons)} persons to cropped data.")


In [ ]:
# need to add some school enrollment data to land use or else size terms are zero
for col in ['ENROLL_K8','ENROLL_912','ENROLL_COLL']:
    if cropped_landuse[col].sum() == 0:
        print(f"Adding {col} data to land use.")
        # sample 10% of the zones to have enrollment
        enrollment_zones = cropped_landuse.sample(frac=.1).index
        cropped_landuse.loc[enrollment_zones, col] = np.random.randint(100, 1000, size=len(enrollment_zones))


In [ ]:
cropped_landuse.to_csv(os.path.join('metro_data_cropped', 'land_use.csv'), index=False)
cropped_households.to_csv(os.path.join('metro_data_cropped', 'households.csv'), index=False)
cropped_persons.to_csv(os.path.join('metro_data_cropped', 'persons.csv'), index=False)